In [1]:
# =============================================================================
# CELL 1: CONNECTION HEALTH GATE
# =============================================================================

import pyodbc

DSN = "Redshift_prod_new"

try:
    with pyodbc.connect(f"DSN={DSN}", timeout=15) as conn:
        conn.execute("SELECT 1")
    print(f"Redshift connection OK  (DSN={DSN})")
except Exception as e:
    raise RuntimeError(f"Cannot connect to Redshift (DSN={DSN}): {e}")

Redshift connection OK  (DSN=Redshift_prod_new)


In [2]:
# =============================================================================
# CELL 2: IMPORTS AND TABLE CONFIG
# =============================================================================

import concurrent.futures
import pyodbc
import pandas as pd
import time
import os
from IPython.display import display, Markdown

update_tables = True


SAMPLE_ROWS = 5
QUERY_TIMEOUT_SEC = 300

# ---------------------------------------------------------------------------
# Freshness query templates for dateless tables.
# {table} is replaced at runtime with the fully qualified table name.
# ---------------------------------------------------------------------------

LOAN_ID_FRESHNESS = """
    SELECT MAX(cd.application_received_date) AS max_dt
    FROM {table} t
    LEFT JOIN edwnpi.los_deal_current_fact cd ON t.loan_id = cd.loan_id
    WHERE cd.application_received_date >= DATEADD(day, -90, CURRENT_DATE)
"""

ACCT_FRESHNESS = """
    SELECT MAX(cd.application_received_date) AS max_dt
    FROM {table} t
    LEFT JOIN edwnpi.los_deal_current_fact cd ON t.account_number = cd.account_number
    WHERE cd.application_received_date >= DATEADD(day, -90, CURRENT_DATE)
"""

CUST_ID_FRESHNESS = """
    SELECT MAX(cd.application_received_date) AS max_dt
    FROM {table} t
    LEFT JOIN edwnpi.los_deal_current_fact cd ON t.customer_id = cd.pb_customer_id
    WHERE cd.application_received_date >= DATEADD(day, -90, CURRENT_DATE)
"""

CUSTOMERID_FRESHNESS = """
    SELECT MAX(cd.application_received_date) AS max_dt
    FROM {table} t
    LEFT JOIN edwnpi.los_deal_current_fact cd ON t.customerid = cd.pb_customer_id
    WHERE cd.application_received_date >= DATEADD(day, -90, CURRENT_DATE)
"""

DEALER_NUM_FRESHNESS = """
    SELECT MAX(cd.application_received_date) AS max_dt
    FROM {table} t
    LEFT JOIN edwnpi.los_deal_current_fact cd ON t.dealer_number = cd.dealer_number
    WHERE cd.application_received_date >= DATEADD(day, -90, CURRENT_DATE)
"""

DEALER_ID_FRESHNESS = """
    SELECT MAX(cd.application_received_date) AS max_dt
    FROM {table} t
    LEFT JOIN edwnpi.los_deal_current_fact cd ON t.dealerid = cd.dealer_number
    WHERE cd.application_received_date >= DATEADD(day, -90, CURRENT_DATE)
"""

# ---------------------------------------------------------------------------
# Table registry -- volatile sandbox tables first, stable edwnpi tables last.
# ---------------------------------------------------------------------------

TABLES_TO_CHECK = [
    # --- Volatile sandbox tables first ---
    {"table": "sandbox.student_loan_chime_flags",                    "key_col": "loan_id",       "date_col": None,       "used_by": "ULA",                          "freshness_query": LOAN_ID_FRESHNESS},
    {"table": "sandbox.temp_employment_type_ragu",                   "key_col": "account_number","date_col": None,       "used_by": "ULA",                          "freshness_query": ACCT_FRESHNESS},
    {"table": "sandbox.rds_blackbook_rollup",                        "key_col": "account_number","date_col": None,       "used_by": "ULA",                          "freshness_query": ACCT_FRESHNESS},
    {"table": "sandbox.kmx_approvals",                               "key_col": "loan_id",       "date_col": None,       "used_by": "ULA",                          "freshness_query": LOAN_ID_FRESHNESS},
    {"table": "sandbox.kmx_los_new_sp",                              "key_col": "loan_id",       "date_col": None,       "used_by": "ULA",                          "freshness_query": LOAN_ID_FRESHNESS},
    {"table": "sandbox.temp_prov_customer_credit_attributes_ragu",   "key_col": "customerid",    "date_col": None,       "used_by": "ULA",                          "freshness_query": CUSTOMERID_FRESHNESS},
    {"table": "sandbox.temp_los_customer_credit_attributes_ragu",    "key_col": "customer_id",   "date_col": None,       "used_by": "ULA",                          "freshness_query": CUST_ID_FRESHNESS},
    {"table": "sandbox.loan_random_numbers",                         "key_col": "loan_id",       "date_col": None,       "used_by": "ULA",                          "freshness_query": LOAN_ID_FRESHNESS},
    {"table": "sandbox.temp_fraud_ragu",                             "key_col": "loan_id",       "date_col": None,       "used_by": "ULA",                          "freshness_query": LOAN_ID_FRESHNESS},
    {"table": "sandbox.temp_blackbook_values_ragu",                   "key_col": "account_number","date_col": None,       "used_by": "ULA / Recovery",               "freshness_query": ACCT_FRESHNESS},
    {"table": "sandbox.nonkmx_dealer_loss_data",                     "key_col": "dealer_number", "date_col": None,       "used_by": "DLA",                          "freshness_query": DEALER_NUM_FRESHNESS},
    {"table": "sandbox.rds_rec_model_originations",                  "key_col": "account_number","date_col": "con_date", "used_by": "Recovery",                     "freshness_query": None},

    # --- Stable edwnpi tables last ---
    {"table": "edwnpi.los_deal_current_fact",                        "key_col": "account_number","date_col": "book_date","used_by": "Model Scores / ULA",           "freshness_query": None},
    {"table": "edwnpi.dealer_rollup_scd_current",                    "key_col": "dealer_number", "date_col": None,       "used_by": "Model Scores / ULA",           "freshness_query": DEALER_NUM_FRESHNESS},
    {"table": "edwnpi.date_dim",                                     "key_col": "calendar_date", "date_col": "calendar_date", "used_by": "Model Scores / ULA / Recovery","freshness_query": None},
    {"table": "edwnpi.dealer_attributes_pivot",                      "key_col": "dealerid",      "date_col": None,       "used_by": "ULA",                          "freshness_query": DEALER_ID_FRESHNESS},
    {"table": "edwnpi.crm_dealer_dim",                               "key_col": "dealer_number", "date_col": None,       "used_by": "ULA",                          "freshness_query": DEALER_NUM_FRESHNESS},
]

print(f"Tables to check: {len(TABLES_TO_CHECK)}")
print(f"Sample rows per table: {SAMPLE_ROWS}")
print(f"Query timeout: {QUERY_TIMEOUT_SEC}s")

Tables to check: 17
Sample rows per table: 5
Query timeout: 300s


In [3]:
# Parameters
update_tables = True


In [4]:
# =============================================================================
# CELL 3: PARALLEL DISCOVERY PROBE
# =============================================================================

import warnings

def check_table(cfg):
    """Two-pass probe for a single table. Runs in its own thread with its own connection."""
    table = cfg["table"]
    key_col = cfg["key_col"]
    date_col = cfg.get("date_col")
    freshness_query = cfg.get("freshness_query")

    result = {
        "table": table,
        "used_by": cfg["used_by"],
        "reachable": False,
        "total_rows": None,
        "non_null_key_rows": None,
        "sample_df": None,
        "columns": None,
        "max_date": None,
        "recent_rows": None,
        "elapsed_sec": None,
        "error": None,
        "freshness_error": None,
    }

    t0 = time.time()
    try:
        conn = pyodbc.connect(f"DSN={DSN}", timeout=15)
        conn.timeout = QUERY_TIMEOUT_SEC
    except Exception as e:
        result["error"] = f"Connection failed: {str(e)[:300]}"
        result["elapsed_sec"] = round(time.time() - t0, 2)
        return result

    try:
        warnings.filterwarnings("ignore", category=UserWarning)

        # ---- PASS 1: Existence, health, and sample (no joins) ----
        count_q = f"SELECT COUNT(*) AS total_rows, COUNT({key_col}) AS non_null_key_rows FROM {table}"
        count_row = pd.read_sql_query(count_q, conn)
        result["reachable"] = True
        result["total_rows"] = int(count_row["total_rows"].iloc[0])
        result["non_null_key_rows"] = int(count_row["non_null_key_rows"].iloc[0])

        sample_q = f"SELECT * FROM {table} LIMIT {SAMPLE_ROWS}"
        sample_df = pd.read_sql_query(sample_q, conn)
        result["sample_df"] = sample_df
        result["columns"] = list(sample_df.columns)

        # ---- PASS 2: Freshness + recent volume (only if Pass 1 succeeded) ----
        try:
            if date_col:
                fresh_q = (
                    f"SELECT COUNT(*) AS recent_rows, MAX({date_col}) AS max_dt FROM {table} "
                    f"WHERE {date_col} >= DATEADD(day, -90, CURRENT_DATE)"
                )
                fresh_row = pd.read_sql_query(fresh_q, conn)
                result["max_date"] = str(fresh_row["max_dt"].iloc[0])
                result["recent_rows"] = int(fresh_row["recent_rows"].iloc[0])
            elif freshness_query:
                fresh_tmpl = freshness_query.replace("MAX(cd.application_received_date) AS max_dt",
                                                     "COUNT(*) AS recent_rows, MAX(cd.application_received_date) AS max_dt")
                fresh_q = fresh_tmpl.format(table=table)
                fresh_row = pd.read_sql_query(fresh_q, conn)
                result["max_date"] = str(fresh_row["max_dt"].iloc[0])
                result["recent_rows"] = int(fresh_row["recent_rows"].iloc[0])
        except Exception as e:
            result["freshness_error"] = f"Freshness check failed (join dependency may be down): {str(e)[:300]}"

        warnings.filterwarnings("default", category=UserWarning)

    except Exception as e:
        result["error"] = str(e)[:300]
    finally:
        conn.close()
        result["elapsed_sec"] = round(time.time() - t0, 2)

    return result


# ---- Dispatch all table checks in parallel ----
print(f"Probing {len(TABLES_TO_CHECK)} tables in parallel ...\n")
probe_start = time.time()

with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(check_table, cfg): cfg["table"] for cfg in TABLES_TO_CHECK}
    probe_results = []
    for future in concurrent.futures.as_completed(futures):
        r = future.result()
        tag = "OK" if r["reachable"] else "FAIL"
        print(f"  [{tag}] {r['table']}  ({r['elapsed_sec']}s)")
        probe_results.append(r)

probe_elapsed = round(time.time() - probe_start, 2)
print(f"\nAll probes complete in {probe_elapsed}s")
print("[PROGRESS] Probe Complete")

Probing 17 tables in parallel ...



  [OK] sandbox.student_loan_chime_flags  (2.37s)


  [OK] sandbox.temp_employment_type_ragu  (3.61s)
  [OK] sandbox.temp_prov_customer_credit_attributes_ragu  (3.64s)


  [OK] sandbox.kmx_los_new_sp  (5.14s)


C:\Users\ahmed.ali\AppData\Local\Temp\ipykernel_37040\2601932032.py:67: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fresh_row = pd.read_sql_query(fresh_q, conn)


...

  [OK] edwnpi.dealer_attributes_pivot  (5.45s)


C:\Users\ahmed.ali\AppData\Local\Temp\ipykernel_37040\2601932032.py:67: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fresh_row = pd.read_sql_query(fresh_q, conn)


  [OK] edwnpi.crm_dealer_dim  (7.17s)


C:\Users\ahmed.ali\AppData\Local\Temp\ipykernel_37040\2601932032.py:60: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fresh_row = pd.read_sql_query(fresh_q, conn)


  [OK] edwnpi.los_deal_current_fact  (40.16s)

All probes complete in 45.63s
[PROGRESS] Probe Complete


In [5]:
# =============================================================================
# CELL 4: SAMPLE DATA VIEWER
# =============================================================================

pd.set_option("display.max_columns", 50)

for r in sorted(probe_results, key=lambda x: x["table"]):
    table = r["table"]
    sample = r["sample_df"]

    display(Markdown(f"---\n### `{table}`"))

    if r["error"]:
        display(Markdown(f"**Error:** {r['error']}"))
        continue

    info_parts = [
        f"**Used by:** {r['used_by']}",
        f"**Rows:** {r['total_rows']:,}",
        f"**Non-null key rows:** {r['non_null_key_rows']:,}",
        f"**Columns ({len(r['columns'])}):** `{'`, `'.join(r['columns'])}`",
        f"**Max date:** {r['max_date']}",
        f"**Elapsed:** {r['elapsed_sec']}s",
    ]
    if r["freshness_error"]:
        info_parts.append(f"**Freshness error:** {r['freshness_error']}")

    display(Markdown("  \n".join(info_parts)))

    if sample is not None and len(sample) > 0:
        display(sample)
    else:
        display(Markdown("*No sample rows returned.*"))

---
### `edwnpi.crm_dealer_dim`

**Used by:** ULA  
**Rows:** 983,922  
**Non-null key rows:** 983,922  
**Columns (152):** `crm_dealer_dim_row_id`, `version_start_useast_dtm`, `version_end_useast_dtm`, `version_number`, `current_version_flag`, `deleted_flag`, `dealer_number`, `aca_advantage_flag`, `activation_useast_dtm`, `ally_dealer_id`, `ally_enabled_flag`, `ancillary_products_flag`, `app_one_enabled_useast_dtm`, `app_one_id`, `auto_nation_wofco_number`, `bulk_product_status_code`, `bulk_product_status_name`, `corporate_dealer_group_code`, `corporate_dealer_group_name`, `creditor_ssn_anomalies_flag`, `daily_stip_report_flag`, `dealer_class_code`, `dealer_class_name`, `dealer_group_code`, `dealer_group_name`, `dealer_make_name_1`, `dealer_make_name_10`, `dealer_make_name_11`, `dealer_make_name_12`, `dealer_make_name_13`, `dealer_make_name_14`, `dealer_make_name_15`, `dealer_make_name_2`, `dealer_make_name_3`, `dealer_make_name_4`, `dealer_make_name_5`, `dealer_make_name_6`, `dealer_make_name_7`, `dealer_make_name_8`, `dealer_make_name_9`, `dealer_track_enabled_flag`, `dealer_track_enabled_useast_dtm`, `dealer_track_id`, `dealer_type_code`, `dealer_type_name`, `dealer_watch_rebate_flag`, `dealer_watch_risk_flag`, `dealer_watch_title_flag`, `document_delivery_email_address`, `document_delivery_preference_code`, `document_delivery_preference_name`, `doing_business_as_name`, `e_contract_ode_submission_flag`, `e_contract_submission_dealer_track_flag`, `enrollment_completion_useast_dtm`, `enrollment_created_useast_dtm`, `fax_number`, `in_activation_date`, `in_activation_reason`, `in_eligible_enrollment_flag`, `inventory_total_amt`, `lead_source_description`, `lead_source_name`, `legal_entity_type_code`, `legal_entity_type_name`, `legal_name`, `loc_product_status_code`, `loc_product_status_name`, `mailing_address_city`, `mailing_address_country_code`, `mailing_address_line_1`, `mailing_address_line_2`, `mailing_address_line_3`, `mailing_address_state_code`, `mailing_address_state_name`, `mailing_address_zip_code`, `market_manager_name`, `market_name`, `new_bulkdealerlevellossadjustmentname`, `new_bulkmarketmanagername`, `new_carsdotcomid`, `new_collateral_swapname`, `new_creditiqenabledname`, `new_creditiqname`, `new_dealrehashname`, `new_dmssystemname`, `new_docgenservice`, `new_dot_team_name`, `new_dotphonenumber`, `new_highlinename`, `new_inventoryqualityname`, `new_lead_originating_id`, `new_leadnumber`, `new_locdealerlevellossadjustmentname`, `new_locmarketmanagername`, `new_lotqualityname`, `new_mmcstatus`, `new_mmcstatusname`, `new_modealersuretybondexpdate`, `new_modealersuretybondname`, `new_motitleprocessname`, `new_posdealeropsagentname`, `new_pricing_aws_flag`, `new_txdocfee`, `new_txocccnotification`, `phone_number`, `physical_address_city`, `physical_address_country_code`, `physical_address_line_1`, `physical_address_line_2`, `physical_address_line_3`, `physical_address_state_code`, `physical_address_state_name`, `physical_address_zip_code`, `poi_or_voe_anomalies_flag`, `pos_funder_name`, `pos_funding_supervisor_name`, `pos_funding_uw_manager_name`, `pos_processing_agent_name`, `pos_product_status_code`, `pos_product_status_name`, `pos_under_writer_name`, `pre_verification_flag`, `pricing_30_pct_down_rule`, `pricing_30_pct_down_rule_flag`, `pricing_delta_mroa_percent`, `pricing_flat_discount_type`, `pricing_hurdle_code`, `pricing_hurdle_name`, `pricing_illuminati_flag`, `pricing_no_flat_fee_flag`, `pricing_no_participation_fee_flag`, `pricing_products_indicator`, `pricing_specialty_dealer_name`, `pricing_tier_1_amt`, `pricing_tier_1_percent`, `pricing_tier_2_amt`, `pricing_tier_2_percent`, `quick_calls_flag`, `rebate_watch_exception_flag`, `rehash_flag`, `risk_dealer_pricing_group_name`, `route_one_enabled_flag`, `route_one_enabled_useast_dtm`, `route_one_id`, `stips_and_documents_flag`, `termination_useast_dtm`, `title_watch_exception_flag`, `used_sales_monthly_amt`, `vehicle_anomalies_flag`, `website`, `disable_rehash_flag`  
**Max date:** 2026-05-12  
**Elapsed:** 7.17s

,crm_dealer_dim_row_id,version_start_useast_dtm,version_end_useast_dtm,version_number,current_version_flag,deleted_flag,dealer_number,aca_advantage_flag,activation_useast_dtm,ally_dealer_id,ally_enabled_flag,ancillary_products_flag,app_one_enabled_useast_dtm,app_one_id,auto_nation_wofco_number,bulk_product_status_code,bulk_product_status_name,corporate_dealer_group_code,corporate_dealer_group_name,creditor_ssn_anomalies_flag,daily_stip_report_flag,dealer_class_code,dealer_class_name,dealer_group_code,dealer_group_name,...,pricing_hurdle_code,pricing_hurdle_name,pricing_illuminati_flag,pricing_no_flat_fee_flag,pricing_no_participation_fee_flag,pricing_products_indicator,pricing_specialty_dealer_name,pricing_tier_1_amt,pricing_tier_1_percent,pricing_tier_2_amt,pricing_tier_2_percent,quick_calls_flag,rebate_watch_exception_flag,rehash_flag,risk_dealer_pricing_group_name,route_one_enabled_flag,route_one_enabled_useast_dtm,route_one_id,stips_and_documents_flag,termination_useast_dtm,title_watch_exception_flag,used_sales_monthly_amt,vehicle_anomalies_flag,website,disable_rehash_flag
0,23,2020-02-11 19:00:00,2020-02-15 19:00:00,1,0,None,171,None,None,None,False,None,None,None,None,None,None,None,None,None,None,None,None,None,None,...,None,None,False,None,None,None,None,None,None,None,None,None,None,None,AN,True,None,None,None,NaT,None,NaN,None,NaN,None
1,55,2020-02-11 19:00:00,2020-02-15 19:00:00,1,0,None,217,None,None,None,False,None,None,None,None,None,None,None,None,None,None,None,None,None,None,...,None,None,False,None,None,None,None,None,None,None,None,None,None,None,AN,True,None,None,None,NaT,None,NaN,None,NaN,None
2,87,2020-02-11 19:00:00,2020-04-05 20:00:00,1,0,None,350,None,None,None,False,None,None,None,None,None,None,None,None,None,None,None,None,None,None,...,None,None,False,None,None,None,None,None,None,None,None,None,None,None,ACA,True,None,None,None,2016-06-15,None,NaN,None,NaN,None
3,119,2020-02-11 19:00:00,2020-04-05 20:00:00,1,0,None,356,None,None,None,False,None,None,None,None,None,None,None,None,None,None,None,None,None,None,...,None,None,False,None,None,None,None,None,None,None,None,None,None,None,ACA,True,None,None,None,NaT,None,NaN,None,NaN,None
4,151,2020-02-11 19:00:00,2020-02-15 19:00:00,1,0,None,392,None,None,None,False,None,None,None,None,None,None,None,None,None,None,None,None,None,None,...,None,None,False,None,None,None,None,None,None,None,None,None,None,None,AN,True,None,None,None,NaT,None,60.0,None,http://www.bobdanielscars.com/Staff,None


---
### `edwnpi.date_dim`

**Used by:** Model Scores / ULA / Recovery  
**Rows:** 14,981  
**Non-null key rows:** 14,981  
**Columns (62):** `date_id`, `calendar_date`, `julian_date`, `calendar_year`, `calendar_quarter`, `calendar_quarter_id`, `calendar_quarter_name`, `season`, `calendar_month_id`, `calendar_month_number`, `month_name`, `month_abbreviation`, `calendar_week_id`, `calendar_week`, `day_of_calendar_year`, `day_of_month`, `week_day_number`, `week_day_name`, `week_day_abbreviation`, `is_week_end_day`, `is_public_holiday`, `is_leap_year`, `is_special_day`, `days_in_calendar_year`, `weekdays_in_calendar_year`, `workdays_in_calendar_year`, `days_in_calendar_year_so_far`, `weekdays_in_calendar_year_so_far`, `workdays_in_calendar_year_so_far`, `days_in_month`, `weekdays_in_month`, `workdays_in_month`, `days_in_month_so_far`, `weekdays_in_month_so_far`, `workdays_in_month_so_far`, `month_first_day_id`, `month_last_day_id`, `week_first_day_id`, `week_last_day_id`, `bi_weekly_first_day_id`, `bi_weekly_last_day_id`, `calendar_quarter_first_day_id`, `calendar_quarter_last_day_id`, `is_last_day_of_month`, `is_first_day_of_month`, `date_prev_id`, `date_next_id`, `calendar_month_prev_id`, `calendar_month_next_id`, `work_week`, `work_week_day_number`, `work_week_first_day_id`, `work_week_last_day_id`, `work_week_id`, `serial_date`, `calendar_half`, `calendar_half_id`, `calendar_half_name`, `calendar_half_first_day_id`, `calendar_half_last_day_id`, `semi_month_first_day_id`, `semi_month_last_day_id`  
**Max date:** 2030-12-31  
**Elapsed:** 4.63s

...

**Used by:** ULA  
**Rows:** 8,218,515  
**Non-null key rows:** 8,218,515  
**Columns (10):** `customer_id`, `dq_auto`, `vantage`, `fico`, `secured_credit_card`, `chime_indicator`, `num_tradelines`, `auth_tradelines`, `prev_chargeoff`, `open_tradelines`  
**Max date:** 2026-05-12  
**Elapsed:** 5.49s

,customer_id,dq_auto,vantage,fico,secured_credit_card,chime_indicator,num_tradelines,auth_tradelines,prev_chargeoff,open_tradelines
0,275,0,612,513,1,1,8,0,0,1
1,665,0,558,502,0,0,27,0,0,12
2,701,0,574,479,0,0,50,1,0,22
3,829,0,540,532,0,0,9,0,0,2
4,830,0,657,657,0,0,6,0,0,1


---
### `sandbox.temp_prov_customer_credit_attributes_ragu`

**Used by:** ULA  
**Rows:** 26,243,057  
**Non-null key rows:** 26,243,057  
**Columns (8):** `customerid`, `dq_auto`, `secured_credit_card`, `num_tradelines`, `auth_tradelines`, `prev_chargeoff`, `open_tradelines`, `prov_chime`  
**Max date:** 2026-05-12  
**Elapsed:** 3.64s

,customerid,dq_auto,secured_credit_card,num_tradelines,auth_tradelines,prev_chargeoff,open_tradelines,prov_chime
0,22891212,0,0,16,0,0,5,0.0
1,22891886,0,0,9,0,0,0,0.0
2,22891758,0,1,37,0,0,22,0.0
3,22892121,0,0,24,0,0,13,0.0
4,22895656,0,1,7,0,0,1,0.0


In [6]:
# =============================================================================
# CELL 5: QUERY FILE EXISTENCE CHECK
# =============================================================================

QUERY_FILES = [
    ("postmodern_ms_query.txt",    "Model Scores"),
    ("vintage_level_ula_query.txt", "ULA"),
    ("new_dll_query.txt",           "DLA"),
    ("new_recovery_queryt.txt",     "New Recovery"),
]

print("Query file check:")
for filename, label in QUERY_FILES:
    exists = os.path.exists(filename)
    full_path = os.path.abspath(filename)
    status = "EXISTS" if exists else "MISSING"
    print(f"  [{status}]  {filename}  ({label})")
    if not exists:
        print(f"           Expected at: {full_path}")

Query file check:
  [EXISTS]  postmodern_ms_query.txt  (Model Scores)
  [EXISTS]  vintage_level_ula_query.txt  (ULA)
  [EXISTS]  new_dll_query.txt  (DLA)
  [EXISTS]  new_recovery_queryt.txt  (New Recovery)


In [7]:
# =============================================================================
# CELL 6: SUMMARY DATAFRAME
# =============================================================================

summary_rows = []
for r in probe_results:
    summary_rows.append({
        "table": r["table"],
        "used_by": r["used_by"],
        "reachable": r["reachable"],
        "total_rows": r["total_rows"],
        "non_null_key_rows": r["non_null_key_rows"],
        "num_columns": len(r["columns"]) if r["columns"] else None,
        "max_date": r["max_date"],
        "recent_rows": r["recent_rows"],
        "elapsed_sec": r["elapsed_sec"],
        "error": r["error"],
        "freshness_error": r["freshness_error"],
    })

diag_df = pd.DataFrame(summary_rows)
diag_df = diag_df.sort_values("elapsed_sec", ascending=False).reset_index(drop=True)

display(Markdown("### Diagnostic Summary"))
display(diag_df)

### Diagnostic Summary

,table,used_by,reachable,total_rows,non_null_key_rows,num_columns,max_date,recent_rows,elapsed_sec,error,freshness_error
0,edwnpi.los_deal_current_fact,Model Scores / ULA,True,32684437,2477902,495,2026-05-12,59811,40.16,None,None
1,sandbox.loan_random_numbers,ULA,True,30595693,30595693,24,2026-05-12,1556667,8.26,None,None
2,edwnpi.crm_dealer_dim,ULA,True,983922,983922,152,2026-05-12,64166121,7.17,None,None
3,sandbox.kmx_approvals,ULA,True,10166853,10166853,26,2026-05-12,797738,5.76,None,None
4,sandbox.temp_los_customer_credit_attributes_ragu,ULA,True,8218515,8218515,10,2026-05-12,1510284,5.49,None,None
5,sandbox.rds_blackbook_rollup,ULA,True,30442908,1238392,10,2026-05-09,56859,5.46,None,None
6,edwnpi.dealer_attributes_pivot,ULA,True,26911,26911,61,2026-05-12,1796456,5.45,None,None
7,sandbox.kmx_los_new_sp,ULA,True,15217396,15217396,8,None,0,5.14,None,None
8,sandbox.nonkmx_dealer_loss_data,DLA,True,622120,622120,37,2026-05-12,62717810,4.82,None,None
9,edwnpi.date_dim,Model Scores / ULA / Recovery,True,14981,14981,62,2030-12-31,1785,4.63,None,None


In [8]:
# =============================================================================
# CELL 7: GUARDRAILS -- STATUS LABELS, STALENESS, COLOR-CODED SUMMARY
# =============================================================================

from datetime import datetime, timedelta

# ---------------------------------------------------------------------------
# Staleness thresholds (days). Tables with own date or freshness join are
# checked against these. Tables with no freshness mechanism are skipped.
# ---------------------------------------------------------------------------
STALENESS_THRESHOLDS = {
    "edwnpi.los_deal_current_fact":  3,
    "sandbox.rds_rec_model_originations": 3,
    "edwnpi.dealer_rollup_scd_current": 3,
    "edwnpi.crm_dealer_dim": 3,
    "edwnpi.dealer_attributes_pivot": 3,
    "edwnpi.date_dim": 30,

    "sandbox.student_loan_chime_flags": 7,
    "sandbox.temp_employment_type_ragu": 7,
    "sandbox.rds_blackbook_rollup": 7,
    "sandbox.kmx_approvals": 7,
    "sandbox.kmx_los_new_sp": 7,
    "sandbox.temp_prov_customer_credit_attributes_ragu": 7,
    "sandbox.temp_los_customer_credit_attributes_ragu": 7,
    "sandbox.loan_random_numbers": 7,
    "sandbox.temp_fraud_ragu": 7,
    "sandbox.temp_blackbook_values_ragu": 7,
    "sandbox.nonkmx_dealer_loss_data": 7,
}

# Tables that SHOULD have recent freshness data (flag if max_date is None)
EXPECTS_FRESHNESS = set(STALENESS_THRESHOLDS.keys())

today = pd.Timestamp.today().normalize()

def compute_status(row):
    if not row["reachable"]:
        return "DOWN"
    if row["total_rows"] == 0:
        return "EMPTY"
    if row["freshness_error"]:
        return "FRESHNESS_ERROR"

    table = row["table"]
    max_date_str = row["max_date"]

    if table in EXPECTS_FRESHNESS:
        if max_date_str in (None, "None", "NaT", "nan"):
            return "NO_FRESHNESS"
        try:
            max_dt = pd.Timestamp(max_date_str)
            days_stale = (today - max_dt).days
            threshold = STALENESS_THRESHOLDS.get(table, 7)
            if days_stale > threshold:
                return f"STALE ({days_stale}d)"
        except Exception:
            return "NO_FRESHNESS"

    return "OK"

guardrail_df = diag_df.copy()
guardrail_df["status"] = guardrail_df.apply(compute_status, axis=1)

# Reorder for readability
display_cols = ["status", "table", "used_by", "max_date", "recent_rows",
                "total_rows", "non_null_key_rows", "elapsed_sec",
                "error", "freshness_error"]
guardrail_df = guardrail_df[display_cols].sort_values(
    "status", key=lambda s: s.map(lambda v: 0 if v != "OK" else 1)
).reset_index(drop=True)

# ---------------------------------------------------------------------------
# Color-code by status
# ---------------------------------------------------------------------------
def highlight_row(row):
    status = row["status"]
    if status == "DOWN":
        return ["background-color: #d32f2f; color: white"] * len(row)
    elif status == "EMPTY":
        return ["background-color: #f57c00; color: white"] * len(row)
    elif status.startswith("STALE"):
        return ["background-color: #ffa726; color: black"] * len(row)
    elif status in ("NO_FRESHNESS", "FRESHNESS_ERROR"):
        return ["background-color: #ffee58; color: black"] * len(row)
    return [""] * len(row)

# ---------------------------------------------------------------------------
# Top-level verdict
# ---------------------------------------------------------------------------
statuses = set(guardrail_df["status"])
blockers = {s for s in statuses if s in ("DOWN", "EMPTY")}
warnings_set = {s for s in statuses if s.startswith("STALE") or s in ("NO_FRESHNESS", "FRESHNESS_ERROR")}

if blockers:
    verdict = "BLOCKED -- critical tables are down or empty. Do NOT run bareboned_ragu_new.ipynb."
    verdict_style = "color: #d32f2f; font-weight: bold; font-size: 16px"
elif warnings_set:
    verdict = "WARNINGS -- some tables are stale or missing freshness data. Review before running."
    verdict_style = "color: #f57c00; font-weight: bold; font-size: 16px"
else:
    verdict = "ALL CLEAR -- all tables are reachable and fresh."
    verdict_style = "color: #2e7d32; font-weight: bold; font-size: 16px"

display(Markdown(f"### Diagnostic Verdict"))
display(Markdown(f'<p style="{verdict_style}">{verdict}</p>'))

if blockers:
    blocked_tables = guardrail_df[guardrail_df["status"].isin(("DOWN", "EMPTY"))]["table"].tolist()
    display(Markdown("**Blocked by:** " + ", ".join(f"`{t}`" for t in blocked_tables)))

if warnings_set:
    warn_mask = guardrail_df["status"].apply(lambda s: s.startswith("STALE") or s in ("NO_FRESHNESS", "FRESHNESS_ERROR"))
    warn_tables = guardrail_df[warn_mask][["table", "status", "max_date"]].to_string(index=False)
    display(Markdown("**Warnings:**\n```\n" + warn_tables + "\n```"))

display(guardrail_df.style.apply(highlight_row, axis=1))
print("[PROGRESS] Guardrails Complete")

### Diagnostic Verdict

<p style="color: #f57c00; font-weight: bold; font-size: 16px">WARNINGS -- some tables are stale or missing freshness data. Review before running.</p>

**Warnings:**
```
                 table       status max_date
sandbox.kmx_los_new_sp NO_FRESHNESS     None
```

,status,table,used_by,max_date,recent_rows,total_rows,non_null_key_rows,elapsed_sec,error,freshness_error
0,NO_FRESHNESS,sandbox.kmx_los_new_sp,ULA,None,0,15217396,15217396,5.140000,None,None
1,OK,edwnpi.los_deal_current_fact,Model Scores / ULA,2026-05-12,59811,32684437,2477902,40.160000,None,None
2,OK,edwnpi.dealer_rollup_scd_current,Model Scores / ULA,2026-05-12,1796445,30110,30110,2.910000,None,None
3,OK,sandbox.temp_fraud_ragu,ULA,2026-05-07,1177046,1457198,1457198,3.360000,None,None
4,OK,sandbox.temp_employment_type_ragu,ULA,2026-05-07,56441,238025,238012,3.610000,None,None
5,OK,sandbox.temp_prov_customer_credit_attributes_ragu,ULA,2026-05-12,1468014,26243057,26243057,3.640000,None,None
6,OK,sandbox.rds_rec_model_originations,Recovery,2026-05-11,54828,1172633,1171888,3.930000,None,None
7,OK,edwnpi.date_dim,Model Scores / ULA / Recovery,2030-12-31,1785,14981,14981,4.630000,None,None
8,OK,sandbox.nonkmx_dealer_loss_data,DLA,2026-05-12,62717810,622120,622120,4.820000,None,None
9,OK,edwnpi.dealer_attributes_pivot,ULA,2026-05-12,1796456,26911,26911,5.450000,None,None


[PROGRESS] Guardrails Complete


In [9]:
# =============================================================================
# CELL 8: UPDATE CONFIGURATION (sandbox table refresh orchestration)
# =============================================================================
#
# Set update_tables = True to refresh the 6 user-owned sandbox tables after
# the diagnostic runs. DDL tables use a staging + atomic rename pattern so
# the public table name is always queryable, even mid-refresh.
#
# Execution order (heaviest DDL first so thread pool slots get claimed by the
# slowest jobs; the stored-procedure dispatch is last because it is cheap
# client-side and its runtime is dominated by server-side work).
# =============================================================================



TEMPTABLES_PATH = "ragu_temptables"

UPDATE_MAX_WORKERS = 5        # lower than probe (8) because DDL is heavy on shared edwnpi.los_deal_current_fact
UPDATE_STMT_TIMEOUT = 1800    # 30 minutes per statement

UPDATE_PLAN = [
    {"table": "sandbox.temp_fraud_ragu",                            "type": "ddl",       "key_col": "loan_id"},
    {"table": "sandbox.temp_blackbook_values_ragu",                 "type": "ddl",       "key_col": "account_number"},
    {"table": "sandbox.temp_los_customer_credit_attributes_ragu",   "type": "ddl",       "key_col": "customer_id"},
    {"table": "sandbox.temp_prov_customer_credit_attributes_ragu",  "type": "ddl",       "key_col": "customerid"},
    {"table": "sandbox.temp_employment_type_ragu",                  "type": "ddl",       "key_col": "account_number"},
    {"table": "sandbox.student_loan_chime_flags",                   "type": "procedure", "key_col": "loan_id",
     "call_sql": "CALL sandbox.student_loan_chime_flags();"},
]

print(f"update_tables = {update_tables}")
print(f"Tables in update plan: {len(UPDATE_PLAN)} "
      f"({sum(1 for e in UPDATE_PLAN if e['type']=='ddl')} DDL, "
      f"{sum(1 for e in UPDATE_PLAN if e['type']=='procedure')} procedure)")
print(f"Source file: {TEMPTABLES_PATH}")
print(f"Max parallel workers: {UPDATE_MAX_WORKERS}")

update_tables = True
Tables in update plan: 6 (5 DDL, 1 procedure)
Source file: ragu_temptables
Max parallel workers: 5


In [10]:
# =============================================================================
# CELL 9: PARSER + STAGING/RENAME UPDATER + PARALLEL DISPATCH
# =============================================================================
#
# For each DDL entry, we:
#   1. Read the original CREATE body from ragu_temptables
#   2. Rewrite the `INTO <table>` clause to target `<table>_new` (staging)
#   3. Run a 10-step sequence per thread:
#        drop_staging -> build_new -> gate_count -> BEGIN -> drop_old ->
#        rename_curr_to_old -> rename_new_to_curr -> COMMIT -> grant -> drop_old_final
#   4. Each step is its own cur.execute() so failures attribute to the exact step.
#
# The stored-procedure entry (student_loan_chime_flags) bypasses all of this
# and runs its single CALL statement.
# =============================================================================

import re
from pathlib import Path


def _find_stmt_terminator(raw: str, start: int) -> int:
    """Scan forward from `start` and return the index of the next semicolon that
    terminates a SQL statement, respecting single-quoted strings, '' escapes,
    and -- line comments. Returns -1 if no terminator found."""
    i = start
    n = len(raw)
    in_string = False
    in_line_comment = False
    while i < n:
        c = raw[i]
        if in_line_comment:
            if c == "\n":
                in_line_comment = False
        elif in_string:
            if c == "'":
                if i + 1 < n and raw[i + 1] == "'":
                    i += 1  # skip escaped quote ''
                else:
                    in_string = False
        else:
            if c == "'":
                in_string = True
            elif c == "-" and i + 1 < n and raw[i + 1] == "-":
                in_line_comment = True
                i += 1
            elif c == ";":
                return i
        i += 1
    return -1


def extract_create_sql(raw: str, tbl: str) -> str:
    """Isolate the single SELECT INTO <tbl> statement body from ragu_temptables
    and rewrite its INTO target to <tbl>_new for the staging pattern.

    Boundaries:
      left:  end of `DROP TABLE IF EXISTS <tbl>;`
      right: the next statement-terminating ; after `INTO <tbl>` (respects
             single-quoted strings, '' escapes, and -- line comments)

    This avoids dependence on any particular verification-SELECT format
    (some tables use `select top 1 *`, others use custom diagnostic queries)."""
    drop_pat = re.compile(r"DROP\s+TABLE\s+IF\s+EXISTS\s+" + re.escape(tbl) + r"\s*;", re.IGNORECASE)
    drop_m = drop_pat.search(raw)
    if not drop_m:
        raise ValueError(f"Could not find DROP TABLE IF EXISTS {tbl}; in source file")

    into_pat = re.compile(r"INTO\s+" + re.escape(tbl) + r"\b", re.IGNORECASE)
    into_m = into_pat.search(raw, pos=drop_m.end())
    if not into_m:
        raise ValueError(f"Could not find 'INTO {tbl}' clause after DROP in source file")

    term_idx = _find_stmt_terminator(raw, into_m.end())
    if term_idx < 0:
        raise ValueError(f"Could not find statement-terminating ';' for SELECT INTO {tbl}")

    body = raw[drop_m.end(): term_idx + 1].strip()

    body_new = re.sub(
        r"(INTO\s+)" + re.escape(tbl) + r"\b",
        lambda m: m.group(1) + tbl + "_new",
        body,
        flags=re.IGNORECASE,
    )
    if body_new == body:
        raise ValueError(f"INTO {tbl} clause not found in CREATE body for staging rewrite")
    return body_new


def build_statements(entry: dict, raw_file: str) -> list:
    """Expand a single UPDATE_PLAN entry into an ordered list of statement dicts."""
    if entry["type"] == "procedure":
        return [{"step": "call_proc", "sql": entry["call_sql"]}]

    tbl = entry["table"]
    short = tbl.split(".")[-1]
    create_body = extract_create_sql(raw_file, tbl)

    # Atomic swap: `rename_curr_old` defers its commit so both renames land
    # in a single transaction committed at the end of `rename_new_curr`.
    # If `rename_new_curr` fails, the except block's conn.rollback() reverts
    # both renames together and the public name keeps its old data.
    return [
        {"step": "drop_staging",      "sql": f"DROP TABLE IF EXISTS {tbl}_new;"},
        {"step": "build_new",         "sql": create_body},
        {"step": "gate_count",        "sql": f"SELECT COUNT(*) FROM {tbl}_new;", "kind": "scalar"},
        {"step": "drop_old",          "sql": f"DROP TABLE IF EXISTS {tbl}_old;"},
        {"step": "rename_curr_old",   "sql": f"ALTER TABLE {tbl} RENAME TO {short}_old;",
                                      "skip_if_not_exists": tbl,
                                      "defer_commit": True},
        {"step": "rename_new_curr",   "sql": f"ALTER TABLE {tbl}_new RENAME TO {short};"},
        {"step": "grant",             "sql": f"CALL sandbox.util_table_grant('{short}');"},
        {"step": "drop_old_final",    "sql": f"DROP TABLE IF EXISTS {tbl}_old;"},
    ]


def update_table(entry: dict) -> dict:
    """Run one table's update sequence in its own connection. Returns a result dict."""
    t0 = time.time()
    result = {
        "table": entry["table"],
        "type": entry["type"],
        "ok": False,
        "failed_step": None,
        "steps_completed": [],
        "error": None,
        "gate_count": None,
        "elapsed_sec": None,
    }

    try:
        conn = pyodbc.connect(f"DSN={DSN}", timeout=15)
        conn.timeout = UPDATE_STMT_TIMEOUT
        conn.autocommit = False
    except Exception as e:
        result["error"] = f"Connection failed: {str(e)[:300]}"
        result["failed_step"] = "connect"
        result["elapsed_sec"] = round(time.time() - t0, 2)
        return result

    cur = conn.cursor()
    current_step = None
    try:
        for step in entry["statements"]:
            current_step = step["step"]

            if step.get("skip_if_not_exists"):
                schema, name = step["skip_if_not_exists"].split(".")
                cur.execute(
                    "SELECT 1 FROM pg_catalog.pg_tables WHERE schemaname = ? AND tablename = ?",
                    schema, name,
                )
                if cur.fetchone() is None:
                    result["steps_completed"].append(f"{current_step} (skipped, no prior table)")
                    continue

            if step.get("kind") == "scalar":
                cur.execute(step["sql"])
                val = cur.fetchone()[0]
                result["steps_completed"].append(f"{current_step}={val}")
                if current_step == "gate_count":
                    result["gate_count"] = int(val) if val is not None else 0
                    if result["gate_count"] == 0:
                        cur.execute(f"DROP TABLE IF EXISTS {entry['table']}_new;")
                        conn.commit()
                        raise RuntimeError("gate_count returned 0 rows; swap aborted, old table preserved")
            else:
                cur.execute(step["sql"])
                # Skip commit for steps flagged defer_commit; they stay in the
                # open transaction until a following step commits them atomically.
                if not step.get("defer_commit"):
                    conn.commit()
                result["steps_completed"].append(current_step)

        result["ok"] = True
    except Exception as e:
        result["error"] = f"{type(e).__name__}: {str(e)[:500]}"
        result["failed_step"] = current_step
        try:
            conn.rollback()
        except Exception:
            pass
    finally:
        try:
            cur.close()
            conn.close()
        except Exception:
            pass

    result["elapsed_sec"] = round(time.time() - t0, 2)
    return result


# ---------------------------------------------------------------------------
# Dispatch (only runs when update_tables is True)
# ---------------------------------------------------------------------------
if update_tables:
    raw_file = Path(TEMPTABLES_PATH).read_text(encoding="utf-8")

    for entry in UPDATE_PLAN:
        entry["statements"] = build_statements(entry, raw_file)

    total_steps = {e["table"]: len(e["statements"]) for e in UPDATE_PLAN}
    print(f"Prepared {len(UPDATE_PLAN)} tables for update. Per-table step counts: {total_steps}\n")

    print(f"Running updates in parallel (max_workers={UPDATE_MAX_WORKERS}) ...\n")
    upd_start = time.time()

    update_results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=UPDATE_MAX_WORKERS) as executor:
        futures = {executor.submit(update_table, e): e["table"] for e in UPDATE_PLAN}
        for future in concurrent.futures.as_completed(futures):
            r = future.result()
            tag = "OK" if r["ok"] else f"FAIL@{r['failed_step']}"
            gate = f" gate={r['gate_count']}" if r.get("gate_count") is not None else ""
            print(f"  [{tag}] {r['table']}  ({r['elapsed_sec']}s){gate}")
            if not r["ok"]:
                print(f"           error: {r['error']}")
            update_results.append(r)

    upd_elapsed = round(time.time() - upd_start, 2)
    print(f"\nAll updates complete in {upd_elapsed}s")
else:
    print("update_tables = False  ->  skipping refresh. "
          "Set update_tables = True in Cell 8 and rerun from there to refresh the 6 sandbox tables.")
    update_results = []

Prepared 6 tables for update. Per-table step counts: {'sandbox.temp_fraud_ragu': 8, 'sandbox.temp_blackbook_values_ragu': 8, 'sandbox.temp_los_customer_credit_attributes_ragu': 8, 'sandbox.temp_prov_customer_credit_attributes_ragu': 8, 'sandbox.temp_employment_type_ragu': 8, 'sandbox.student_loan_chime_flags': 1}

Running updates in parallel (max_workers=5) ...



  [OK] sandbox.temp_employment_type_ragu  (14.87s) gate=239899


  [OK] sandbox.temp_fraud_ragu  (454.34s) gate=1475467


  [OK] sandbox.temp_prov_customer_credit_attributes_ragu  (619.23s) gate=26243057


  [OK] sandbox.temp_blackbook_values_ragu  (632.77s) gate=1640030


  [OK] sandbox.temp_los_customer_credit_attributes_ragu  (634.71s) gate=8283994


  [OK] sandbox.student_loan_chime_flags  (1161.35s)

All updates complete in 1176.21s


In [11]:
# =============================================================================
# CELL 10: POST-UPDATE INTEGRITY PROBE (moderate)
# =============================================================================
#
# Reuses the same check_table() probe from Cell 3 on just the 6 updated tables.
# Each probe runs in its own thread (one connection per table) so the whole
# integrity pass completes in ~one slowest-table's worth of wall clock time.
#
# Captures: row_count, non_null_key_rows, max_date, recent_rows, elapsed,
#           and any per-table freshness_error / probe error.
#
# Skipped entirely if update_tables = False.
# =============================================================================

if update_tables and update_results:
    updated_tables = {r["table"] for r in update_results}

    probe_cfgs = [cfg for cfg in TABLES_TO_CHECK if cfg["table"] in updated_tables]
    if len(probe_cfgs) != len(updated_tables):
        missing = updated_tables - {c["table"] for c in probe_cfgs}
        print(f"  WARNING: {len(missing)} updated tables have no matching TABLES_TO_CHECK config "
              f"and will be skipped in the integrity probe: {sorted(missing)}")

    print(f"Running post-update integrity probe on {len(probe_cfgs)} tables ...\n")
    iprobe_start = time.time()

    integrity_results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=len(probe_cfgs) or 1) as executor:
        futures = {executor.submit(check_table, cfg): cfg["table"] for cfg in probe_cfgs}
        for future in concurrent.futures.as_completed(futures):
            r = future.result()
            tag = "OK" if r["reachable"] else "FAIL"
            print(f"  [{tag}] {r['table']}  ({r['elapsed_sec']}s)  "
                  f"rows={r['total_rows']}  max_date={r['max_date']}")
            integrity_results.append(r)

    iprobe_elapsed = round(time.time() - iprobe_start, 2)
    print(f"\nIntegrity probe complete in {iprobe_elapsed}s")

    upd_df = pd.DataFrame([
        {
            "table": r["table"],
            "type": r["type"],
            "update_ok": r["ok"],
            "failed_step": r["failed_step"],
            "gate_count": r["gate_count"],
            "update_elapsed_sec": r["elapsed_sec"],
            "update_error": r["error"],
        } for r in update_results
    ])

    int_df = pd.DataFrame([
        {
            "table": r["table"],
            "reachable_post": r["reachable"],
            "post_total_rows": r["total_rows"],
            "post_non_null_key_rows": r["non_null_key_rows"],
            "post_max_date": r["max_date"],
            "post_recent_rows": r["recent_rows"],
            "post_probe_error": r["error"],
            "post_freshness_error": r["freshness_error"],
        } for r in integrity_results
    ])

    update_summary_df = upd_df.merge(int_df, on="table", how="left")

    if "post_non_null_key_rows" in update_summary_df.columns and "post_total_rows" in update_summary_df.columns:
        update_summary_df["post_key_null_pct"] = (
            (update_summary_df["post_total_rows"] - update_summary_df["post_non_null_key_rows"])
            / update_summary_df["post_total_rows"].replace(0, pd.NA) * 100
        ).round(2)

    display_cols = ["table", "type", "update_ok", "failed_step", "gate_count",
                    "update_elapsed_sec", "post_total_rows", "post_key_null_pct",
                    "post_max_date", "post_recent_rows",
                    "update_error", "post_probe_error", "post_freshness_error"]
    existing_cols = [c for c in display_cols if c in update_summary_df.columns]
    update_summary_df = update_summary_df[existing_cols]

    display(Markdown("### Update + Integrity Summary"))
    display(update_summary_df)
else:
    update_summary_df = None
    print("Skipped -- update_tables = False or no update results to probe.")

Running post-update integrity probe on 6 tables ...



  [OK] sandbox.temp_employment_type_ragu  (2.58s)  rows=239899  max_date=2026-05-11


C:\Users\ahmed.ali\AppData\Local\Temp\ipykernel_37040\2601932032.py:67: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fresh_row = pd.read_sql_query(fresh_q, conn)


  [OK] sandbox.student_loan_chime_flags  (2.79s)  rows=33244146  max_date=2026-05-12
  [OK] sandbox.temp_blackbook_values_ragu  (2.95s)  rows=1640030  max_date=2026-05-11


  [OK] sandbox.temp_fraud_ragu  (3.37s)  rows=1475467  max_date=2026-05-12


C:\Users\ahmed.ali\AppData\Local\Temp\ipykernel_37040\2601932032.py:67: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fresh_row = pd.read_sql_query(fresh_q, conn)


  [OK] sandbox.temp_los_customer_credit_attributes_ragu  (6.41s)  rows=8283994  max_date=2026-05-12
  [OK] sandbox.temp_prov_customer_credit_attributes_ragu  (6.49s)  rows=26243057  max_date=2026-05-12

Integrity probe complete in 6.5s


### Update + Integrity Summary

,table,type,update_ok,failed_step,gate_count,update_elapsed_sec,post_total_rows,post_key_null_pct,post_max_date,post_recent_rows,update_error,post_probe_error,post_freshness_error
0,sandbox.temp_employment_type_ragu,ddl,True,None,239899.0,14.87,239899,0.04,2026-05-11,58338,None,None,None
1,sandbox.temp_fraud_ragu,ddl,True,None,1475467.0,454.34,1475467,0.00,2026-05-12,1214987,None,None,None
2,sandbox.temp_prov_customer_credit_attributes_ragu,ddl,True,None,26243057.0,619.23,26243057,0.00,2026-05-12,1468014,None,None,None
3,sandbox.temp_blackbook_values_ragu,ddl,True,None,1640030.0,632.77,1640030,0.00,2026-05-11,58332,None,None,None
4,sandbox.temp_los_customer_credit_attributes_ragu,ddl,True,None,8283994.0,634.71,8283994,0.00,2026-05-12,1565675,None,None,None
5,sandbox.student_loan_chime_flags,procedure,True,None,NaN,1161.35,33244146,0.00,2026-05-12,1731734,None,None,None


In [12]:
# =============================================================================
# CELL 11: UPDATE VERDICT (color-coded, step-level attribution)
# =============================================================================

if update_tables and update_summary_df is not None and not update_summary_df.empty:
    today_upd = pd.Timestamp.today().normalize()

    def _update_status(row):
        if not row.get("update_ok"):
            step = row.get("failed_step") or "unknown"
            if step in ("connect", "drop_staging", "build_new"):
                return "BUILD_FAILED"
            if step == "gate_count":
                return "GATE_FAILED"
            if step in ("drop_old", "rename_curr_old", "rename_new_curr"):
                return "SWAP_FAILED"
            if step == "grant":
                return "GRANT_FAILED"
            if step == "drop_old_final":
                return "CLEANUP_WARNING"
            if step == "call_proc":
                return "PROC_FAILED"
            return f"FAILED@{step}"

        if row.get("post_probe_error"):
            return "POST_PROBE_ERROR"
        if row.get("post_total_rows") in (None, 0) or pd.isna(row.get("post_total_rows")):
            return "UPDATE_SUCCESS_BUT_EMPTY"

        max_date_str = row.get("post_max_date")
        threshold = STALENESS_THRESHOLDS.get(row["table"], 7)
        if max_date_str not in (None, "None", "NaT", "nan") and not pd.isna(max_date_str):
            try:
                max_dt = pd.Timestamp(max_date_str)
                days_stale = (today_upd - max_dt).days
                if days_stale > threshold:
                    return f"UPDATE_SUCCESS_BUT_STALE ({days_stale}d)"
            except Exception:
                pass

        return "UPDATE_OK"

    verdict_df = update_summary_df.copy()
    verdict_df["update_status"] = verdict_df.apply(_update_status, axis=1)

    lead_cols = ["update_status", "table", "type", "failed_step", "gate_count",
                 "update_elapsed_sec", "post_total_rows", "post_key_null_pct",
                 "post_max_date"]
    tail_cols = [c for c in verdict_df.columns if c not in lead_cols + ["update_status"]]
    verdict_df = verdict_df[lead_cols + tail_cols]
    verdict_df = verdict_df.sort_values(
        "update_status", key=lambda s: s.map(lambda v: 0 if v != "UPDATE_OK" else 1)
    ).reset_index(drop=True)

    BLOCKERS = {"BUILD_FAILED", "GATE_FAILED", "SWAP_FAILED", "PROC_FAILED",
                "UPDATE_SUCCESS_BUT_EMPTY", "POST_PROBE_ERROR"}
    WARNINGS = {"GRANT_FAILED", "CLEANUP_WARNING"}

    def _row_style(row):
        s = row["update_status"]
        if s in BLOCKERS or s.startswith("FAILED@"):
            return ["background-color: #d32f2f; color: white"] * len(row)
        if s in WARNINGS:
            return ["background-color: #f57c00; color: white"] * len(row)
        if s.startswith("UPDATE_SUCCESS_BUT_STALE"):
            return ["background-color: #ffa726; color: black"] * len(row)
        return [""] * len(row)

    statuses = set(verdict_df["update_status"])
    blocker_hits = {s for s in statuses if s in BLOCKERS or s.startswith("FAILED@")}
    warning_hits = {s for s in statuses if s in WARNINGS or s.startswith("UPDATE_SUCCESS_BUT_STALE")}

    if blocker_hits:
        verdict_msg = "UPDATE BLOCKED -- one or more tables failed or produced empty/unhealthy output. Review before running bareboned_ragu_new.ipynb."
        verdict_color = "#d32f2f"
    elif warning_hits:
        verdict_msg = "UPDATE WARNINGS -- tables refreshed but grants, cleanup, or freshness have issues. Review before relying on them."
        verdict_color = "#f57c00"
    else:
        verdict_msg = "ALL UPDATES CLEAR -- all 6 tables refreshed and verified."
        verdict_color = "#2e7d32"

    display(Markdown("### Update Verdict"))
    display(Markdown(
        f'<p style="color: {verdict_color}; font-weight: bold; font-size: 16px">{verdict_msg}</p>'
    ))

    if blocker_hits:
        blocked = verdict_df[verdict_df["update_status"].apply(
            lambda s: s in BLOCKERS or s.startswith("FAILED@"))]["table"].tolist()
        display(Markdown("**Blocked / failed:** " + ", ".join(f"`{t}`" for t in blocked)))

    if warning_hits:
        warned = verdict_df[verdict_df["update_status"].apply(
            lambda s: s in WARNINGS or s.startswith("UPDATE_SUCCESS_BUT_STALE"))][
            ["table", "update_status", "post_max_date"]].to_string(index=False)
        display(Markdown("**Warnings:**\n```\n" + warned + "\n```"))

    display(verdict_df.style.apply(_row_style, axis=1))
else:
    print("Skipped -- update_tables = False or nothing to summarize.")
print("[PROGRESS] Diagnostic Complete")

### Update Verdict

<p style="color: #2e7d32; font-weight: bold; font-size: 16px">ALL UPDATES CLEAR -- all 6 tables refreshed and verified.</p>

,update_status,table,type,failed_step,gate_count,update_elapsed_sec,post_total_rows,post_key_null_pct,post_max_date,update_ok,post_recent_rows,update_error,post_probe_error,post_freshness_error
0,UPDATE_OK,sandbox.temp_employment_type_ragu,ddl,None,239899.000000,14.870000,239899,0.040000,2026-05-11,True,58338,None,None,None
1,UPDATE_OK,sandbox.temp_fraud_ragu,ddl,None,1475467.000000,454.340000,1475467,0.000000,2026-05-12,True,1214987,None,None,None
2,UPDATE_OK,sandbox.temp_prov_customer_credit_attributes_ragu,ddl,None,26243057.000000,619.230000,26243057,0.000000,2026-05-12,True,1468014,None,None,None
3,UPDATE_OK,sandbox.temp_blackbook_values_ragu,ddl,None,1640030.000000,632.770000,1640030,0.000000,2026-05-11,True,58332,None,None,None
4,UPDATE_OK,sandbox.temp_los_customer_credit_attributes_ragu,ddl,None,8283994.000000,634.710000,8283994,0.000000,2026-05-12,True,1565675,None,None,None
5,UPDATE_OK,sandbox.student_loan_chime_flags,procedure,None,nan,1161.350000,33244146,0.000000,2026-05-12,True,1731734,None,None,None


[PROGRESS] Diagnostic Complete
